# 🎯 [ISR02_M03] 게시물 유사도 기반 단순 복제본 탐지 및 1차 출처 분류 Baseline 알고리즘 구현
### Google Colab 실습 환경 (정보·감시·정찰(ISR) / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/ISR02/isr02_m03_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 정보·감시·정찰(ISR) (Level 2) |
| **미션 ID** | ISR02_M03 |
| **미션 주제** | 게시물 유사도 기반 단순 복제본 탐지 및 1차 출처 분류 Baseline 알고리즘 구현 |
| **소요 시간** | 약 2.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 텔레그램·X 같은 SNS에 현장 제보와 함께 재게시, 수치·지역 변조, 과거 사진 재활용, 생성형 AI 조작물, 조직적 동시 유포가 한꺼번에 쏟아지면서 공개출처정보(OSINT)의 출처와 신뢰도를 빠르게 가려내는 일이 정보전의 핵심이 되었다. 본 프로젝트에서는 가상 계정과 가상 지역 코드로 만든 교육용 합성 게시물을 사용해 원천 데이터 스키마 정의, 텍스트 정제와 이미지 지각 해시 기반 중복 제거, 유사도 기반 복제·1차 출처 분류 Baseline, 조작·기만 탐지와 계정 신뢰도 결합 모델, 종합 신뢰도 등급(A~E) 엔진, 정보전 모니터링 대시보드, 정보관 긴급 검토(HITL), 대량 스트림 부하 시험, AAR 재보정, 최종 통합까지 단계별로 구현한다. 최종 목표는 조작 게시물을 놓치지 않으면서 사실 제보를 가짜로 몰지 않고, 파급력이 큰 첩보의 채택·차단은 원천 출처를 확인한 정보관이 결정하도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 지휘통제체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/isr02_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/` | 전처리, Baseline, 앙상블 탐지, 대시보드, HITL 등 핵심 파이썬 소스 코드 모듈 |
| **`docs/`** | `docs/isr02_data_spec.md`, `docs/isr02_hitl_guide.md`, `docs/isr02_aar_report.md`, `docs/isr02_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서, 최종 브리핑 등 군사 행정/지휘관 문서 |
| **`configs/`**| `configs/isr02_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/isr02_stress_test.py` | 500% 부하 시험 및 단위/통합 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/preprocess/isr02_preprocessor.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 03. 게시물 유사도 기반 단순 복제본 탐지 및 1차 출처 분류 Baseline 알고리즘 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

게시물 간 텍스트/이미지 유사도 기반으로 단순 복제 및 인용 관계를 판별하는 1차 출처 분류 Baseline 알고리즘을 개발한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **게시물 간 텍스트·이미지 유사도로 단순 복제와 인용(재작성) 관계를 판별하고 1차 출처를 분류하는 Baseline 알고리즘을 개발한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **정제 본문의 문자 n-gram TF-IDF 코사인 유사도와 pHash 거리로 복제·재작성·무관 관계를 규칙 판정하고, 재작성 판단 기준을 바꿔 가며 정확도를 1차 조정 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/models/isr02_baseline_model.py 및 기본 추론 결과 보고서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 알고리즘의 논리적 흐름 및 군사적 판단 기준의 투명성

#### 미션 PASS 조건

- 정량 기준: 기본 데이터셋 대상 탐지/처리 정확도 75% 이상 달성


- 진행 기준: 단위 기능 동작 확인 및 기본 시나리오 실행 로그 생성


- 검증 방법: 기준 벤치마크 데이터셋 대상 추론 결과 오차율 비교


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""게시물 쌍의 본문 유사도(문자 n-gram TF-IDF 코사인)와 이미지 지각 해시 거리로 단순 복제·재작성(인용)·무관 관계를 판별하고, 같은 이야기로 묶인 게시물 중 가장 이른 것을 1차 출처로 분류하는 Baseline 알고리즘을 구현해 정확도와 처리 지연을 측정한다."""
import re
import time
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.fft import dctn
from sklearn.feature_extraction.text import TfidfVectorizer

# ---------------------------------------------------------------------------
# [교육용가상자료] 합성 SNS 게시물 생성기 (실제 계정·채널·지명·게시물 없음)
# 가상 사건마다 원본 제보, 재게시(복제), 유사 재작성, 조작·기만 게시물 10종과 32x32 합성 이미지를 만든다.
# ---------------------------------------------------------------------------
REGIONS = [f"{a}-{n}" for a in "ABCDEF" for n in range(1, 6)]
OBJECTS = ["군용 차량", "소형 드론", "보급 트럭", "헬기", "버스", "견인 차량", "트레일러", "민간 승용차"]
DIRS = ["북쪽", "남쪽", "동쪽", "서쪽"]
TEMPLATES = ["가상 {r} 지역에서 {o} 목격, {n}대 규모로 {d}으로 이동 중",
             "{r} 일대 {o} 확인됨. 약 {n}대, {d} 방향으로 이동",
             "[현지] {r}에서 {o} 발견 - 수량 {n}, 이동 방향 {d}",
             "{r} 주민 제보: {o} {n}대가 {d}으로 향하는 모습"]
MANIP = ["num_change", "region_swap", "dir_flip", "old_image", "splice", "ai_image", "ai_text", "bot_burst", "mashup", "recycled"]
ACCOUNT_TYPES = {  # 유형별 계정 속성 분포(가상): 나이(일), 팔로워, 하루 게시 수, 원본 비율, 과거 정확도, 게시 간격 변동계수
    "media": ((1000, 4000), (5e4, 5e5), (5, 30), (.6, .9), (.85, .97), (.8, 1.5)),
    "witness": ((10, 800), (20, 2000), (.2, 3), (.7, 1.0), (.6, .95), (1.0, 2.0)),
    "aggregator": ((200, 2000), (1e4, 2e5), (30, 200), (.05, .2), (.6, .85), (.5, 1.0)),
    "propaganda": ((5, 200), (100, 2e4), (20, 150), (.3, .7), (.1, .4), (.05, .3)),
}


def make_accounts(rng, n=(8, 30, 10, 20)):
    acc = {}
    for (t, spec), k in zip(ACCOUNT_TYPES.items(), n):
        for i in range(k):
            v = [rng.uniform(*r) for r in spec]
            acc[f"ACC_{t[:3].upper()}{i:02d}"] = {"type": t, "age_days": int(v[0]), "followers": int(v[1]), "posts_per_day": round(v[2], 1),
                                                   "original_ratio": round(v[3], 2), "past_accuracy": round(v[4], 2), "interval_cv": round(v[5], 2)}
    return acc


def event_image(seed):
    """사건 현장 사진을 흉내 낸 32x32 합성 이미지(사건마다 고유한 도형 배치)."""
    # [TODO : 학생 실습 3] 게시물 유사도 기반 단순 복제본 탐지 및 1차 출처 분류 Baseline 알고리즘 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def variant(img, rng):
    """재게시 과정의 자연스러운 변형: 재압축 잡음, 밝기 변화, 가장자리 잘림 중 하나."""
    # [TODO : 학생 실습 3] 게시물 유사도 기반 단순 복제본 탐지 및 1차 출처 분류 Baseline 알고리즘 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def splice(img, other, rng):
    out = img.copy()
    y, x = rng.integers(4, 18, 2)
    out[y:y + 12, x:x + 12] = other[y:y + 12, x:x + 12] + rng.normal(0, 12, (12, 12))   # 잡음 수준이 다른 조각을 붙임
    return out


def ai_image(rng):
    """생성형 모델 이미지를 흉내 낸 합성 이미지: 지나치게 매끈한 바탕 + 업샘플링 격자 무늬(주기 2픽셀)."""
    base = ndimage.gaussian_filter(rng.normal(130, 60, (32, 32)), 3)
    grid = (np.indices((32, 32)).sum(0) % 2) * rng.uniform(6, 12)
    return base + grid


def write(ev, rng, tpl=None, **kw):
    s = {"r": ev["r"], "o": ev["o"], "n": ev["n"], "d": ev["d"], **kw}
    return (tpl or TEMPLATES[int(rng.integers(len(TEMPLATES)))]).format(**s)


def make_posts(n_events=40, seed=7):
    rng = np.random.default_rng(seed)
    acc = make_accounts(rng)
    by = {t: [a for a, v in acc.items() if v["type"] == t] for t in ACCOUNT_TYPES}
    events = [{"id": f"EV{i:03d}", "r": str(rng.choice(REGIONS)), "o": str(rng.choice(OBJECTS)), "n": int(rng.integers(2, 15)),
               "d": str(rng.choice(DIRS)), "t0": float(i * 1.2 + rng.uniform(0, 1)), "img": event_image(seed * 1000 + i)} for i in range(n_events)]
    posts = []

    def add(ev, account, text, img, t, fwd, kind, origin):
        pid = f"P{len(posts):04d}"
        posts.append({"post_id": pid, "platform": str(rng.choice(["telegram", "x", "other"], p=[.6, .3, .1])), "account_id": account,
                      "posted_at": round(t, 3), "text": text, "image": img, "forwarded_from": fwd, "event_id": ev["id"] if ev else None,
                      "kind": kind, "is_disinfo": kind in MANIP, "origin": origin})
        return pid

    for k, ev in enumerate(events):
        src = str(rng.choice(by["witness"] if rng.random() < .6 else by["media"]))
        if rng.random() < .05:
            src = str(rng.choice(by["propaganda"]))                      # 신뢰도 낮은 계정도 사실을 올릴 수 있다
        o = add(ev, src, write(ev, rng), ev["img"], ev["t0"], None, "original", None)
        for _ in range(int(rng.integers(2, 5))):                          # 그대로 재게시(전달)
            add(ev, str(rng.choice(by["aggregator"])), posts[int(o[1:])]["text"], ev["img"], ev["t0"] + rng.exponential(1), o, "copy", o)
        for _ in range(int(rng.integers(1, 3))):                          # 표현만 바꾼 재작성(내용 동일)
            add(ev, str(rng.choice(by["media"] + by["witness"])), "#가상속보 " + write(ev, rng), variant(ev["img"], rng),
                ev["t0"] + rng.uniform(0.5, 6), None, "near_dup", o)
        for m in MANIP if k % 2 == 0 else MANIP[k % 4::4]:                # 조작·기만 게시물
            if rng.random() > 0.5:
                continue
            who = str(rng.choice(by["propaganda"] if rng.random() < .75 else by["aggregator"] + by["witness"]))
            t, img, text = ev["t0"] + rng.uniform(1, 12), variant(ev["img"], rng), None
            if m == "num_change":
                text = write(ev, rng, n=ev["n"] * int(rng.integers(5, 10)))
            elif m == "region_swap":
                text = write(ev, rng, r=str(rng.choice([r for r in REGIONS if r != ev["r"]])))
            elif m == "dir_flip":
                text = write(ev, rng, d={"북쪽": "남쪽", "남쪽": "북쪽", "동쪽": "서쪽", "서쪽": "동쪽"}[ev["d"]])
            elif m == "old_image" and k > 0:
                text, img = write(ev, rng), events[int(rng.integers(0, k))]["img"]
            elif m == "splice":
                text, img = write(ev, rng), splice(ev["img"], events[(k + 7) % n_events]["img"], rng)
            elif m == "ai_image":
                text, img = write(ev, rng), ai_image(rng)
            elif m == "ai_text":
                text = f"속보 속보! {ev['r']} 지역 대규모 {ev['o']} 출현! 긴급 확산 바람! 긴급 확산 바람! 모두 확산 바람!"
                img = ai_image(rng) if rng.random() < .5 else None
            elif m == "bot_burst":
                text = write(ev, rng, n=ev["n"] * 6) + " 지금 바로 공유"
                for b in rng.choice(by["propaganda"], 4, replace=False):
                    add(ev, str(b), text, img, t + rng.uniform(0, 0.15), None, m, o)
                continue
            elif m == "mashup":
                other = events[(k + 3) % n_events]
                text = f"{ev['r']} 일대 {ev['o']} 확인, {other['n']}대가 {other['d']}으로 이동"   # 다른 사건 내용과 짜깁기
            elif m == "recycled" and k > 2:
                old = events[int(rng.integers(0, k - 1))]
                text, img, t = write(old, rng), old["img"], ev["t0"] + rng.uniform(0, 3)
            if text is None:
                continue
            add(ev, who, text, img, t, None, m, o)
    return posts, acc
# ---------------------------------------------------------------------------

# ---- 텍스트 정제·이미지 지각 해시·조작 흔적 공통 함수 ----
STOPWORDS = {"가상", "약", "일대", "지역에서", "모습", "중", "확인됨", "발견", "제보", "주민"}
_URL = re.compile(r"https?://\S+|www\.\S+")
_SPECIAL = re.compile(r"[^0-9A-Za-z가-힣\-\s]")


def clean_text(t):
    """정규화(NFKC) -> URL·멘션 제거 -> 해시태그 기호 제거 -> 특수문자 제거 -> 불용어 제거 -> 공백 정리."""
    t = unicodedata.normalize("NFKC", t)
    t = _URL.sub(" ", t)
    t = re.sub(r"@\w+", " ", t).replace("#", " ")
    t = _SPECIAL.sub(" ", t)
    return " ".join(w for w in t.split() if w not in STOPWORDS)


def phash(img):
    """지각 해시(pHash): 32x32 이미지 DCT의 저주파 8x8(직류 성분 제외)을 중앙값과 비교한 64비트."""
    c = dctn(np.asarray(img, float), norm="ortho")[:8, :8].ravel()[1:]
    return np.append(c > np.median(c), False)


def hamming(a, b):
    return int(np.count_nonzero(a != b))


def slots(text):
    """본문에서 지역 코드·수량·방향·대상을 뽑는다(사건 내용 비교용)."""
    r = re.search(r"[A-F]-\d", text)
    n = re.search(r"(\d+)\s*대|수량\s*(\d+)", text)
    d = next((x for x in DIRS if x in text), None)
    o = next((x for x in OBJECTS if x in text), None)
    return {"r": r.group(0) if r else None, "n": int(n.group(1) or n.group(2)) if n else None, "d": d, "o": o}


def splice_score(img):
    """합성(붙여넣기) 흔적: 8x8 블록별 고주파 잡음 크기의 최대/중앙값 비율. 잡음 수준이 다른 조각이 있으면 커진다."""
    res = img - ndimage.median_filter(img, 3)
    b = np.abs(res).reshape(4, 8, 4, 8).std(axis=(1, 3))
    return float(b.max() / (np.median(b) + 1e-6))


def ai_image_score(img):
    """생성형 이미지 흔적: 2픽셀 주기 격자(나이퀴스트 주파수) 에너지 비율."""
    f = np.abs(np.fft.fft2(img - img.mean()))
    return float(f[16, 16] / (f.sum() / f.size + 1e-6))


def ai_text_score(text):
    """기계적 선동 문체 흔적: 반복 어절 비율 + 느낌표 밀도."""
    w = text.split()
    return (1 - len(set(w)) / max(len(w), 1)) + text.count("!") / max(len(w), 1)


def relation(sim, dist, R):
    if sim >= R["copy_sim"] and (dist is None or dist <= R["copy_ham"]):
        return "복제"
    if sim >= R["quote_sim"] or (dist is not None and dist <= R["quote_ham"]):
        return "재작성"
    return "무관"


R0 = {"copy_sim": 0.95, "copy_ham": 6, "quote_sim": 0.45, "quote_ham": 10}
posts, acc = make_posts(60, seed=11)
true_posts = [p for p in posts if not p["is_disinfo"]]
idx = {p["post_id"]: i for i, p in enumerate(posts)}
X = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)).fit_transform([clean_text(p["text"]) for p in posts])
H = [phash(p["image"]) if p["image"] is not None else None for p in posts]
dist = lambda i, j: hamming(H[i], H[j]) if H[i] is not None and H[j] is not None else None

# 1) 평가용 게시물 쌍: 복제(전달), 재작성, 무관(다른 사건)
rng = np.random.default_rng(0)
pairs = [(idx[p["post_id"]], idx[p["origin"]], {"copy": "복제", "near_dup": "재작성"}[p["kind"]]) for p in true_posts if p["kind"] in ("copy", "near_dup")]
while sum(1 for *_, l in pairs if l == "무관") < 150:
    i, j = rng.integers(0, len(posts), 2)
    if posts[i]["event_id"] != posts[j]["event_id"] and not posts[i]["is_disinfo"] and not posts[j]["is_disinfo"]:
        pairs.append((int(i), int(j), "무관"))
t0 = time.perf_counter()
sims = [(X[i] @ X[j].T).toarray()[0, 0] for i, j, _ in pairs]
pred = [relation(s, dist(i, j), R0) for s, (i, j, _) in zip(sims, pairs)]
lat = (time.perf_counter() - t0) / len(pairs) * 1000
truth = [l for *_, l in pairs]
acc0 = np.mean([a == b for a, b in zip(pred, truth)])
print(f"게시물 쌍 {len(pairs)}개 관계 판별 정확도 {acc0:.1%} (목표 75% 이상: {'달성' if acc0 >= 0.75 else '미달'}), 쌍당 {lat:.2f} ms")
print(pd.crosstab(pd.Series(truth, name="실제"), pd.Series(pred, name="판별")).to_string())

# 2) 휴리스틱 가중치 1차 튜닝: 재작성 판단 유사도 기준
print("\n재작성 기준(유사도)  정확도")
for q in [0.3, 0.45, 0.6, 0.75]:
    pr = [relation(s, dist(i, j), {**R0, "quote_sim": q}) for s, (i, j, _) in zip(sims, pairs)]
    print(f"      {q:.2f}         {np.mean([a == b for a, b in zip(pr, truth)]):.1%}")

# 3) 1차 출처 분류: 전달 경로(forwarded_from)를 거슬러 올라가고, 경로가 없으면 같은 이야기 중 가장 이른 게시물을 출처로 본다
order = sorted(range(len(true_posts)), key=lambda k: true_posts[k]["posted_at"])
tp = [idx[p["post_id"]] for p in true_posts]
src, label = {}, {}
for k in order:
    i = tp[k]; p = posts[i]
    if p["forwarded_from"]:
        j = idx[p["forwarded_from"]]
        while posts[j]["forwarded_from"]:
            j = idx[posts[j]["forwarded_from"]]
        src[i], label[i] = j, "전달"
        continue
    cand = [j for j in src if src[j] == j and relation((X[i] @ X[j].T).toarray()[0, 0], dist(i, j), R0) != "무관"]
    if cand:
        src[i], label[i] = src[min(cand, key=lambda j: posts[j]["posted_at"])], "재작성"
    else:
        src[i], label[i] = i, "1차 출처"
true_lab = {"original": "1차 출처", "copy": "전달", "near_dup": "재작성"}
ok_lab = np.mean([label[i] == true_lab[posts[i]["kind"]] for i in tp])
ok_src = np.mean([posts[src[i]]["post_id"] == (posts[i]["origin"] or posts[i]["post_id"]) for i in tp])
print(f"\n1차 출처 분류(사실 게시물 {len(tp)}건): 유형 정확도 {ok_lab:.1%}, 출처 게시물 일치율 {ok_src:.1%}")
print("출처 일치율이 낮은 이유: 전달 경로가 없는 재작성 글은 같은 문형을 쓴 다른 사건 글과 혼동된다(본문 유사도만 사용).")
print("한계: 조작 게시물은 원본과 표현이 비슷해 '재작성'으로 묶이며, 내용 변조 여부는 판단하지 못한다(미션 4의 동기).")

plt.figure(figsize=(7, 4))
for lab, c in [("복제", "tab:red"), ("재작성", "tab:orange"), ("무관", "tab:gray")]:
    sel = [k for k, t in enumerate(truth) if t == lab]
    plt.scatter([sims[k] for k in sel], [dist(*pairs[k][:2]) if dist(*pairs[k][:2]) is not None else -4 for k in sel], s=12, c=c,
                label={"복제": "copy", "재작성": "rewrite", "무관": "unrelated"}[lab])
plt.axvline(R0["quote_sim"], ls="--", c="k"); plt.axhline(R0["quote_ham"], ls=":", c="k")
plt.xlabel("text cosine similarity"); plt.ylabel("pHash distance (-4 = no image)"); plt.legend(); plt.title("Pair relation baseline (synthetic posts)")
plt.show()


<details>
<summary><b>💡 [참고] 미션 03 예상 실행 결과 (Output)</b></summary>

```text
게시물 쌍 421개 관계 판별 정확도 98.3% (목표 75% 이상: 달성), 쌍당 0.22 ms
판별    무관   복제  재작성
실제                
무관   143    0    7
복제     0  178    0
재작성    0    0   93

재작성 기준(유사도)  정확도
      0.30         93.3%
      0.45         98.3%
      0.60         97.9%
      0.75         96.9%

1차 출처 분류(사실 게시물 331건): 유형 정확도 85.5%, 출처 게시물 일치율 64.0%
출처 일치율이 낮은 이유: 전달 경로가 없는 재작성 글은 같은 문형을 쓴 다른 사건 글과 혼동된다(본문 유사도만 사용).
한계: 조작 게시물은 원본과 표현이 비슷해 '재작성'으로 묶이며, 내용 변조 여부는 판단하지 못한다(미션 4의 동기).
```
</details>
